#### 영상콘텐츠산업과 K콘텐츠 플랫폼 조성관련 국내외 방문객 현황분석

In [12]:
import pandas as pd
import geopandas as gpd
import glob

# 경로와 기존 데이터 변수설정
data_path = "D:\\02_사업관리\\2026년\\05_사업관리\\데이터 분석 컨설팅\\20260119_영상콘텐츠산업과\\"
code_path = "D:\\02_사업관리\\코드자료\\행정경계코드\\jscode20260102\\"
gdf1 = gpd.read_file(data_path + "2026-001_데이터 분석 컨설팅 데이터.gpkg", layer="월별_위치_시간대_방문인구현황_100m")
grid100 = gpd.read_file(data_path + "2026-001_데이터 분석 컨설팅 데이터.gpkg", layer="sgis_busan_100m")

In [13]:
pd.set_option('display.max_row', 500)

kikcd_h = pd.read_excel(code_path + "KIKcd_H.20260102.xlsx", dtype={'행정동코드': str}, engine='openpyxl')
# print(kikcd_h.shape) # 3920
sdf = kikcd_h[(kikcd_h['읍면동명'].isnull()) & (~kikcd_h['행정동코드'].str.startswith('26'))]

# 울산, 경남 외 타시도
sdf1 = sdf[(~sdf['시도명'].str.contains('부산|울산|경상남도|출장소')) & (sdf['시군구명'].isnull())]
sdf1 = sdf1[['행정동코드', '시도명']]
sdf1.loc[sdf1['시도명'].str.contains('서울특별시'), '유입지명'] = '서울'
sdf1.loc[sdf1['시도명'].str.contains('경기도'), '유입지명'] = '경기'
sdf1.loc[sdf1['시도명'].str.contains('광역시'), '유입지명'] = sdf1['시도명'].str[:2]
sdf1.loc[sdf1['시도명'].str.contains('북도'), '유입지명'] = sdf1['시도명'].str[:1] + "북"
sdf1.loc[sdf1['시도명'].str.contains('남도'), '유입지명'] = sdf1['시도명'].str[:1] + "북"
sdf1.loc[sdf1['시도명'].str.contains('자치'), '유입지명'] = sdf1['시도명'].str[:2]
sdf1.columns = ['inflow_cd', 'sname', 'inflow_nm']
# print(sdf1)

# 울산, 경남(시군구까지 포함)
sdf2 = sdf[(sdf['시도명'].str.contains('부산|울산|경상남도')) & (~sdf['시군구명'].isnull())].copy()
sdf2 = sdf2[~sdf2['시군구명'].str.contains('출장소')]
sdf2.loc[sdf2['시도명'].str.contains('광역시'), '유입지명'] = sdf2['시도명'].str[:2] + ' ' + sdf2['시군구명']
sdf2.loc[sdf2['시도명'].str.contains('남도'), '유입지명'] = sdf2['시도명'].str[:1] + '남 ' + sdf2['시군구명']
sdf2 = sdf2[['행정동코드', '시도명', '유입지명']]
sdf2.columns = ['inflow_cd', 'sname', 'inflow_nm']
# print(sdf2)

# sdf_all = pd.concat([sdf1, sdf2], ignore_index=True)
# sdf_all.columns = ['inflow_cd', 'sname', 'inflow_nm']

In [ ]:
# 100m격자 유입지별
values = []
data_lists = glob.glob(data_path + "집계결과\\kt_d_busan_event_2025\\*.csv", recursive=True)
for data in data_lists:
    values.append(pd.read_csv(data, sep="|", dtype={'STD_YM': str, 'INFLOW_CD': str, 'TME': str}, encoding="utf-8").rename(columns=str.lower))
    df = pd.concat(values, ignore_index=True)
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x_axis, df.y_axis), crs=5179)

# 부울경을 기준으로 타시도와 부울경으로 분리 -> 데이터 최적화, 좌표컬럼 제외(x_axis, y_axis)
# 타시도 유입인구
gdf1 = gdf[gdf['inflow_cd'].isin([inflow for inflow in sdf1['inflow_cd']])]
gdf1 = pd.merge(gdf1, sdf1, "left", on='inflow_cd')
gdf1 = gdf1[['std_ym','gid','inflow_nm','tme','total','geometry']]

# 부울경 유입인구
gdf2 = gdf[gdf['inflow_cd'].isin([inflow for inflow in sdf2['inflow_cd']])]
gdf2 = pd.merge(gdf2, sdf2, "left", on='inflow_cd')
gdf2 = gdf2[['std_ym','gid','inflow_cd','tme','total','geometry']]

# GPKG로 저장
gdf1.to_file(data_path + "2026-001_데이터 분석 컨설팅 데이터.gpkg", layer="월별_100m_시간대_유입인구(타시도)현황", driver="GPKG")
gdf2.to_file(data_path + "2026-001_데이터 분석 컨설팅 데이터.gpkg", layer="월별_100m_시간대_유입인구(부울경)현황", driver="GPKG")

KeyError: '행정동코드'

In [ ]:
gdf1 = gdf1[['std_ym','gid','inflow_nm','tme','total','geometry']]

# 부울경 유입인구
gdf2 = gdf[gdf['inflow_cd'].isin([inflow for inflow in sdf2['행정동코드']])]
gdf2 = gdf1[['std_ym','gid','inflow_cd','tme','total','geometry']]

# GPKG로 저장
gdf1.to_file(data_path + "2026-001_데이터 분석 컨설팅 데이터.gpkg", layer="월별_100m_시간대_유입인구(타시도)현황", driver="GPKG")
gdf2.to_file(data_path + "2026-001_데이터 분석 컨설팅 데이터.gpkg", layer="월별_100m_시간대_유입인구(부울경)현황", driver="GPKG")

In [ ]:
gdf2 = pd.merge(grid100, df, "left", "gid").dropna()
inner = gdf[gdf['inflow_cd'].str.startswith('26')].copy()
outer = gdf[~gdf['inflow_cd'].str.startswith('26')].co
inner = inner[['std_ym','gid','tme','inflow_cd','total','geometry']]aa
inner = pd.merge(inner, sdf_all, "left", "inflow_cd")
inner = inner[['std_ym', 'gid', 'tme', 'inflow_cd', 'inflow_nm', 'total', 'geometry']]
inner.to_file(data_path + "2026-001_데이터 분석 컨설팅 데이터.gpkg", layer="월별_100m_시간대_유입인구(내지인)", driver="GPKG")
print("월별 100m 시간대 유입인구(내지인) 저장완료")
outer = outer[['std_ym','gid','tme','inflow_cd','total','geometry']]
outer = pd.merge(outer, sdf_all, "left", "inflow_cd")
outer = outer[['std_ym', 'gid', 'tme', 'inflow_cd', 'inflow_nm', 'total', 'geometry']]
outer.to_file(data_path + "2026-001_데이터 분석 컨설팅 데이터.gpkg", layer="월별_100m_시간대_유입인구(외지인)", driver="GPKG")
print("월별 100m 시간대 유입인구(외지인) 저장완료")